# 04 · Payer-equity audit

**Question:** is the decline decision associated with payer — and does any raw difference survive adjustment for case mix?

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

# Anchor on this project specifically: it sits in a subdirectory of a repository
# that has its own pyproject.toml, so "nearest pyproject.toml" is not enough.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "transfer_decline").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))


## Two tests
1. **Chi-square** tests of independence: payer × decision, and med-surg occupancy band × decision (a check that the test picks up the association the generator *does* build in).
2. **Multivariable logistic regression** of `declined` on payer, adjusting for acuity, ICU and med-surg occupancy, ED boarding, service line, and level of care. This separates 'this payer is declined more' from 'this payer's requests are sicker / arrive when the hospital is full'.

In [2]:
df = pd.read_csv(ROOT / 'data' / 'synthetic' / 'prepared_transfer_requests.csv')
from transfer_decline.equity import chi_square_tests, decline_rate_by_payer, payer_logit, payer_logit_summary
display(decline_rate_by_payer(df))

,payer,n,declines,rate
0,Medicaid,1509,269,0.178264
1,Medicare,3810,622,0.163255
2,Unknown,263,41,0.155894
3,Commercial,2877,440,0.152937
4,Self-Pay / Uninsured,541,81,0.149723


In [3]:
chi = chi_square_tests(df)
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'table'} for k, v in chi.items()}, indent=2))

{
  "payer_vs_decision": {
    "chi2": 5.3923945632906,
    "p_value": 0.24935124544855392,
    "dof": 4,
    "cramers_v": 0.024477641869198378,
    "n": 9000,
    "significant_at_0.05": false
  },
  "medsurg_occupancy_vs_decision": {
    "chi2": 799.6587423955827,
    "p_value": 2.2714876815872533e-174,
    "dof": 2,
    "cramers_v": 0.29807880061702974,
    "n": 9000,
    "significant_at_0.05": true
  }
}


The occupancy test is significant — as it should be, since capacity drives declines in the generator. The payer test is the one under audit.

In [4]:
display(payer_logit(df))
summary = payer_logit_summary(df)
print(json.dumps({k: v for k, v in summary.items() if k != 'odds_ratios'}, indent=2))

,payer,reference,odds_ratio,ci_low,ci_high,p_value
0,Medicaid,Commercial,1.184889,0.988816,1.419843,0.066047
1,Medicare,Commercial,1.081422,0.935978,1.249467,0.288156
2,Self-Pay / Uninsured,Commercial,0.953703,0.719915,1.263413,0.741116


{
  "reference_payer": "Commercial",
  "n_complete_case": 8601,
  "adjustment_terms": [
    "acuity_score",
    "icu_occupancy_pct",
    "medsurg_occupancy_pct",
    "ed_boarding_count",
    "C(requested_service_line)",
    "C(requested_level_of_care)"
  ],
  "any_payer_term_significant_at_0.05": false,
  "reading": "No payer term is significant after adjusting for case mix; consistent with payer being unrelated to the decision in this synthetic data."
}


## Reading it
In this synthetic data payer is assigned independently of the decision, so the audit comes back null: no payer term is significant after adjustment. The model in notebook 05 still **excludes payer entirely** — a clean audit today is not a guarantee for tomorrow, and a model that never sees payer cannot learn to use it. That is a design safeguard, not a reaction to a finding.

On real data this same audit could show a disparity; it would then be a starting point for investigation, not a conclusion about intent.